# Food-101 data and deterministic preprocessing — Notebook 02 exports

**Status: authored, not executed.** This notebook contains reusable definitions; importing its `ahr-export` cells performs no download, audit, training, or evaluation. The user requested creation without Run All.

**Purpose:** acquire the official Food-101 archive with checksum verification; audit original decoded images; reserve the official test partition; create deterministic, duplicate-cleaned training/development memberships; provide arbitrary-resolution deterministic preprocessing and masks.

**Inputs:** the official archive (or an explicit future `allow_download=True`), `config['data']`, image/model/preprocessing/masking parameters. **Outputs:** immutable source/split/audit manifests in `data/food101/`, resumable progress in `checkpoints/food101-data-*`, and the dataset bundle used by the Notebook 02 runner. **Expected runtime:** definition import is seconds; initial download/extraction/full audit may take tens of minutes or longer and depends on network/storage. No runtime has been measured for this notebook.

**Exact data resume:** after an interruption, use the same project root and config and call `prepare_food101(root, config, allow_download=...)` again. It verifies the archive and committed progress, reuses verified extraction/audit records, and resumes the first uncommitted chunk. Invalid committed metadata or changed source files raise; they never silently restart. A filesystem lock permits one preparation writer. No model checkpoints are created here.

The official training metadata contains noisy labels by dataset design. Labels determine stratification and later probes, not the JEPA objective. Test images may be decoded only for integrity/duplicate checks; no test predictive evaluation is offered by this adapter. Exact RGB duplicate detection cannot find near duplicates.

Source: [official Food-101 page](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/) and [torchvision source contract](https://docs.pytorch.org/vision/main/_modules/torchvision/datasets/food101.html). The published archive checksum is MD5; this adapter additionally records and verifies SHA256 for the downloaded bytes. HTTPS is used for the official download transport.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


## Source identity and interruption-safe acquisition

These definitions require only the existing standard library, NumPy, Pillow and PyTorch. No torchvision import or dependency installation is required. The official archive is the source of image identities, class names and official splits. Extraction rejects links, special files and paths outside the dataset directory.

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
import fcntl, hashlib, io, json, math, os, re, shutil, tarfile, time, urllib.request, uuid
import numpy as np
import torch
from PIL import Image
from fmm_jepa.data import sample_masks

DATA_SCHEMA = "fmm-jepa-food101-data/v1"
AUDIT_VERSION = "food101-original-decoded-rgb-sha256/v1"
SOURCE_URL = "https://data.vision.ee.ethz.ch/cvl/food-101.tar.gz"
PUBLISHED_URL = "http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz"
SOURCE_MD5 = "85eeb15f3717b99a5da872d97d918f87"
SOURCE_REFERENCE = "https://docs.pytorch.org/vision/main/_modules/torchvision/datasets/food101.html"
DUPLICATE_POLICY = "original decoded RGB groups prioritize official test > seeded development > training; keep lexicographically first identity in the winning partition; exclude all remaining copies before subset selection; do not relabel"


def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")
    try:
        with partial.open("w") as stream:
            json.dump(value, stream, sort_keys=True, indent=2, allow_nan=False)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(partial, path)
        descriptor = os.open(path.parent, os.O_DIRECTORY)
        try:
            os.fsync(descriptor)
        finally:
            os.close(descriptor)
    finally:
        if partial.exists():
            partial.unlink()


def _seal(value):
    value = dict(value)
    value.pop("manifest_sha256", None)
    return {**value, "manifest_sha256": canonical_hash(value)}


def _read_sealed(path):
    value = json.loads(Path(path).read_text())
    if canonical_hash({k: v for k, v in value.items() if k != "manifest_sha256"}) != value.get("manifest_sha256"):
        raise RuntimeError(f"Invalid committed manifest: {path}; inspect before recovery")
    return value


def _save_progress(path, value):
    # Retain the previous committed stage manifest as well as the latest.
    path = Path(path)
    if path.exists():
        previous = _read_sealed(path)
        atomic_json(path.with_name(path.stem + ".previous.json"), previous)
    atomic_json(path, _seal(value))


def stable_seed(base_seed, stream, *identity):
    return int(canonical_hash([int(base_seed), stream, *identity])[:16], 16) % (2**63 - 1)


@contextmanager
def _one_writer(cache):
    cache = Path(cache)
    cache.mkdir(parents=True, exist_ok=True)
    with (cache / ".prepare.lock").open("a+") as stream:
        try:
            fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError("Another Food-101 data preparation writer is active") from exc
        try:
            yield
        finally:
            fcntl.flock(stream, fcntl.LOCK_UN)


def _archive_hashes(path):
    md5, sha = hashlib.md5(), hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            md5.update(block)
            sha.update(block)
    if md5.hexdigest() != SOURCE_MD5:
        raise RuntimeError("Food-101 archive differs from the published checksum; refusing extraction")
    return {"md5": md5.hexdigest(), "sha256": sha.hexdigest(), "bytes": Path(path).stat().st_size}


def _download_archive(root, cache, allow_download):
    path = cache / "food-101.tar.gz"
    if path.exists():
        return path
    if not allow_download:
        raise FileNotFoundError(f"Food-101 archive missing at {path}. Download is disabled; explicitly enable the notebook download setting when ready.")
    if shutil.disk_usage(cache).free < 16 * 1024**3:
        raise RuntimeError("Require at least 16 GiB free for archive, extracted images, audits and temporary writes")
    partial = path.with_suffix(path.suffix + ".partial")
    progress_path = Path(root) / "checkpoints/food101-download/manifest.json"
    previous = _read_sealed(progress_path) if progress_path.exists() else None
    if previous and (previous.get("url") != SOURCE_URL or previous.get("expected_md5") != SOURCE_MD5):
        raise RuntimeError("Download identity changed; explicit recovery required")
    if partial.exists() and previous is None:
        raise RuntimeError("Orphan partial download without a progress manifest; inspect before recovery")
    committed = int(previous.get("committed_bytes", 0)) if previous else 0
    if committed and (not partial.exists() or partial.stat().st_size < committed):
        raise RuntimeError("Committed download bytes are missing")
    if partial.exists():
        with partial.open("r+b") as stream:
            stream.truncate(committed)
    headers = {"User-Agent": "fmm-jepa-research-notebook/0.2"}
    if committed:
        headers["Range"] = f"bytes={committed}-"
    request = urllib.request.Request(SOURCE_URL, headers=headers)
    with urllib.request.urlopen(request, timeout=120) as response:
        resumed = bool(committed and response.status == 206)
        if resumed and not response.headers.get("Content-Range", "").startswith(f"bytes {committed}-"):
            raise RuntimeError("Server returned a mismatched byte range")
        if committed and not resumed:
            # A server without byte-range support requires a new transfer of the
            # same immutable archive. This is explicit in the progress record.
            committed = 0
        state = {"schema": DATA_SCHEMA, "stage": "download", "url": SOURCE_URL,
                 "expected_md5": SOURCE_MD5, "committed_bytes": committed,
                 "resumed_http_range": resumed, "status": "IN_PROGRESS"}
        _save_progress(progress_path, state)
        last_save, last_bytes = time.monotonic(), committed
        with partial.open("ab" if resumed else "wb") as stream:
            while True:
                block = response.read(8 * 1024 * 1024)
                if not block:
                    break
                stream.write(block)
                committed += len(block)
                if committed - last_bytes >= 64 * 1024 * 1024 or time.monotonic() - last_save >= 900:
                    stream.flush()
                    os.fsync(stream.fileno())
                    state["committed_bytes"] = committed
                    _save_progress(progress_path, state)
                    last_save, last_bytes = time.monotonic(), committed
            stream.flush()
            os.fsync(stream.fileno())
    hashes = _archive_hashes(partial)
    os.replace(partial, path)
    _save_progress(progress_path, {**state, "status": "COMPLETE", "committed_bytes": committed, **hashes})
    return path


def _safe_archive_target(cache, member):
    name = PurePosixPath(member.name)
    if (name.is_absolute() or not name.parts or name.parts[0] != "food-101"
            or ".." in name.parts or "\\" in member.name
            or not (member.isfile() or member.isdir())):
        raise RuntimeError(f"Unsafe Food-101 archive member: {member.name!r}")
    destination = cache.joinpath(*name.parts)
    if not destination.resolve().is_relative_to(cache.resolve()) or not destination.resolve().is_relative_to((cache / "food-101").resolve()):
        raise RuntimeError("Archive destination escapes source directory")
    return destination


def _extract_archive(root, cache, archive, hashes):
    checkpoint = Path(root) / "checkpoints" / ("food101-data-" + hashes["sha256"][:16])
    progress_path = checkpoint / "extraction.json"
    shard_folder = cache / "extraction_shards"
    shard_folder.mkdir(parents=True, exist_ok=True)
    state = _read_sealed(progress_path) if progress_path.exists() else {
        "schema": DATA_SCHEMA, "stage": "extraction", "archive_sha256": hashes["sha256"],
        "next_member": 0, "files_committed": 0, "bytes_committed": 0,
        "hash_shards": [], "status": "IN_PROGRESS"}
    if state.get("archive_sha256") != hashes["sha256"]:
        raise RuntimeError("Extraction source identity changed")
    verified_files = 0
    for entry in state["hash_shards"]:
        shard_path = shard_folder / entry["file"]
        if shard_path.name != entry["file"] or file_sha256(shard_path) != entry["sha256"]:
            raise RuntimeError("Committed extraction hash shard is missing or corrupt")
        shard = json.loads(shard_path.read_text())
        if shard["archive_sha256"] != hashes["sha256"]:
            raise RuntimeError("Extraction hash shard source mismatch")
        for item in shard["files"]:
            relative = PurePosixPath(item["path"])
            if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != "food-101":
                raise RuntimeError("Unsafe committed extraction record path")
            destination = cache.joinpath(*relative.parts)
            if not destination.resolve().is_relative_to(cache.resolve()):
                raise RuntimeError("Committed image escapes the source cache")
            if not destination.is_file() or destination.stat().st_size != item["bytes"] or file_sha256(destination) != item["sha256"]:
                raise RuntimeError(f"Committed extracted file missing or changed: {destination}")
            verified_files += 1
    if verified_files != int(state["files_committed"]):
        raise RuntimeError("Extraction manifest counts differ from committed hash records")
    if state["status"] == "COMPLETE":
        return checkpoint
    cursor = int(state["next_member"])
    files, size, last_save = int(state["files_committed"]), int(state["bytes_committed"]), time.monotonic()
    pending = []
    _save_progress(progress_path, state)

    def commit():
        nonlocal pending, last_save
        if pending:
            filename = f"files-{len(state['hash_shards']):05d}.json"
            shard_path = shard_folder / filename
            atomic_json(shard_path, {"archive_sha256": hashes["sha256"], "files": pending})
            state["hash_shards"].append({"file": filename, "sha256": file_sha256(shard_path), "count": len(pending)})
            pending = []
        _save_progress(progress_path, state)
        last_save = time.monotonic()

    with tarfile.open(archive, "r:gz") as source:
        for ordinal, member in enumerate(source):
            destination = _safe_archive_target(cache, member)
            if ordinal < cursor:
                continue
            if member.isdir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                temporary = destination.with_name(destination.name + ".extract.partial")
                digest = hashlib.sha256()
                with source.extractfile(member) as incoming, temporary.open("wb") as outgoing:
                    for block in iter(lambda: incoming.read(1024 * 1024), b""):
                        outgoing.write(block)
                        digest.update(block)
                    outgoing.flush()
                    os.fsync(outgoing.fileno())
                if temporary.stat().st_size != member.size:
                    raise RuntimeError("Truncated archive member")
                os.replace(temporary, destination)
                files += 1
                size += member.size
                pending.append({"path": member.name, "bytes": member.size, "sha256": digest.hexdigest()})
            state.update(next_member=ordinal + 1, files_committed=files, bytes_committed=size)
            if len(pending) >= 250 or time.monotonic() - last_save >= 900:
                commit()
    state["status"] = "COMPLETE"
    commit()
    return checkpoint


def _image_key(value, class_name=None):
    if not isinstance(value, str) or not re.fullmatch(r"[a-z0-9_]+/[A-Za-z0-9_-]+", value):
        raise ValueError(f"Unsafe or malformed Food-101 image identifier: {value!r}")
    if class_name is not None and value.split("/")[0] != class_name:
        raise ValueError("Image path class differs from source metadata class")
    return value


def sample_id(split, image_key):
    if split not in ("train", "test"):
        raise ValueError("Unknown Food-101 source split")
    return f"food101:{split}:{_image_key(image_key)}"


def parse_sample_id(identifier):
    if not isinstance(identifier, str):
        raise ValueError("Sample identity must be a string")
    fields = identifier.split(":")
    if len(fields) != 3 or fields[0] != "food101" or fields[1] not in ("train", "test"):
        raise ValueError("Unknown Food-101 sample identity")
    return fields[1], _image_key(fields[2])


def _source_metadata(cache):
    base = cache / "food-101/meta"
    metadata = {split: json.loads((base / f"{split}.json").read_text()) for split in ("train", "test")}
    names = sorted(metadata["train"])
    if len(names) != 101 or sorted(metadata["test"]) != names:
        raise RuntimeError("Food-101 class mapping differs across official source splits")
    classes_path = base / "classes.txt"
    if classes_path.exists() and sorted(classes_path.read_text().splitlines()) != names:
        raise RuntimeError("Food-101 classes.txt disagrees with official JSON metadata")
    seen, records = set(), {}
    for split, per_class in (("train", 750), ("test", 250)):
        records[split] = []
        for label, name in enumerate(names):
            keys = metadata[split][name]
            if len(keys) != per_class:
                raise RuntimeError(f"Unexpected official {split} source count for {name}")
            for key in sorted(keys):
                key = _image_key(key, name)
                if key in seen:
                    raise RuntimeError("Repeated original image path in official metadata")
                seen.add(key)
                path = cache / "food-101/images" / (key + ".jpg")
                if not path.is_file():
                    raise RuntimeError(f"Missing original image: {path}")
                records[split].append({"sample_id": sample_id(split, key), "key": key, "label": label})
    return names, records, {f"{split}.json": file_sha256(base / f"{split}.json") for split in ("train", "test")}


def ensure_source(root, allow_download=False):
    root = Path(root).resolve()
    cache = root / "data/food101"
    cache.mkdir(parents=True, exist_ok=True)
    archive = _download_archive(root, cache, bool(allow_download))
    hashes = _archive_hashes(archive)
    checkpoint = _extract_archive(root, cache, archive, hashes)
    names, records, metadata_hashes = _source_metadata(cache)
    identity = {"source": "official Food-101 archive", "archive_sha256": hashes["sha256"],
                "class_names": names, "metadata_sha256": metadata_hashes}
    manifest_path = cache / "dataset_manifest.json"
    if manifest_path.exists():
        manifest = _read_sealed(manifest_path)
        if manifest.get("dataset_fingerprint") != canonical_hash(identity):
            raise RuntimeError("Food-101 immutable source manifest differs from current source")
        return cache, manifest, records
    manifest = {"schema": DATA_SCHEMA, "repository": "Food-101 official archive", "source_url": SOURCE_URL,
        "published_source_url": PUBLISHED_URL, "checksum_reference": SOURCE_REFERENCE,
        "source_revision": hashes["sha256"], "archive": {"file": archive.name, **hashes},
        "files": [{"cache_filename": archive.name, "url": SOURCE_URL, **hashes}],
        "class_names": names, "class_mapping_sha256": canonical_hash(names),
        "metadata_sha256": metadata_hashes, "dataset_fingerprint": canonical_hash(identity),
        "source_splits": {split: {"rows": len(rows), "class_counts": dict(Counter(str(r["label"]) for r in rows))} for split, rows in records.items()},
        "heldout_source_split": "test", "heldout_policy": "reserved; integrity decoding only; predictive evaluation NOT_RUN",
        "load_policy": "local original images; no remote code; no torchvision dependency",
        "source_label_caveat": "official training images intentionally include label/visual noise; no automatic relabeling",
        "checkpoint_dir": str(checkpoint), "verified_at_utc": datetime.now(timezone.utc).isoformat()}
    manifest = _seal(manifest)
    atomic_json(manifest_path, manifest)
    return cache, manifest, records


class Food101Adapter:
    """Decode original dimensions on demand; only official metadata IDs are accepted."""
    def __init__(self, cache, manifest, records=None):
        self.cache = Path(cache)
        self.manifest = manifest
        self.class_names = list(manifest["class_names"])
        self.fingerprint = manifest["dataset_fingerprint"]
        if records is None:
            names, records, metadata_hashes = _source_metadata(self.cache)
            if names != self.class_names or metadata_hashes != manifest["metadata_sha256"]:
                raise RuntimeError("Adapter metadata does not match pinned source manifest")
        self.records = records
        self.index = {r["sample_id"]: r for rows in records.values() for r in rows}
        self.source_ids = {split: [r["sample_id"] for r in rows] for split, rows in records.items()}
        self.labels = {split: np.asarray([r["label"] for r in rows], dtype=np.int64) for split, rows in records.items()}

    def path_for(self, identifier):
        parse_sample_id(identifier)
        if identifier not in self.index:
            raise KeyError("Image identity is not in pinned official source metadata")
        path = self.cache / "food-101/images" / (self.index[identifier]["key"] + ".jpg")
        base = (self.cache / "food-101/images").resolve()
        if path.is_symlink() or not path.resolve().is_relative_to(base) or not path.resolve().is_relative_to(self.cache.resolve()):
            raise RuntimeError("Image path escapes the pinned original source")
        return path

    def get(self, identifier):
        split, _ = parse_sample_id(identifier)
        path = self.path_for(identifier)
        try:
            with Image.open(path) as original:
                dimensions, mode = list(original.size), original.mode
                image = original.convert("RGB")
                image.load()
        except Exception as exc:
            raise RuntimeError(f"Original image decoding failed: {identifier}") from exc
        return {"image": image, "label": int(self.index[identifier]["label"]), "sample_id": identifier,
                "source_dimensions": dimensions, "source_mode": mode, "split": split}

    def __len__(self):
        return len(self.index)


## Full integrity audit and fixed memberships

The audit decodes original dimensions before any crop or resize. Every 250 images (or 15 minutes) it atomically commits a verified shard and advances its cursor. Resume checks prior shard and original-file hashes before proceeding. The official test partition receives duplicate priority, followed by development and training. Split sizes are selected after exclusions; insufficient class counts fail explicitly.

Default candidate development pool: 100 per class from official training, selected with seed 314159. Default pilot memberships: 100 training and 20 development per class; diagnostics use 2 fixed development images per class. All source class names and per-image original dimensions are retained in source manifests and audit shards.

In [ ]:
def audit_source(dataset):
    folder = dataset.cache / "audit_shards"
    folder.mkdir(parents=True, exist_ok=True)
    checkpoint = Path(dataset.manifest["checkpoint_dir"])
    progress_path = checkpoint / "decoded_audit.json"
    ids = dataset.source_ids["train"] + dataset.source_ids["test"]
    identity = {"version": AUDIT_VERSION, "dataset_fingerprint": dataset.fingerprint,
                "ordered_ids_sha256": canonical_hash(ids)}
    state = _read_sealed(progress_path) if progress_path.exists() else {
        **identity, "next_index": 0, "shards": [], "status": "IN_PROGRESS"}
    if any(state.get(key) != value for key, value in identity.items()):
        raise RuntimeError("Audit identity changed; refusing silent restart")
    records, cursor = [], 0
    for entry in state["shards"]:
        path = folder / entry["file"]
        if path.name != entry["file"] or not path.exists() or file_sha256(path) != entry["sha256"]:
            raise RuntimeError("Committed original-image audit shard missing or corrupt")
        shard = json.loads(path.read_text())
        if shard["identity"] != identity or shard["first_index"] != cursor:
            raise RuntimeError("Audit shard cursor or source identity mismatch")
        for record in shard["records"]:
            if record["sample_id"] != ids[cursor]:
                raise RuntimeError("Committed audit image order mismatch")
            original = dataset.path_for(record["sample_id"])
            if original.stat().st_size != record["encoded_bytes"] or file_sha256(original) != record["encoded_sha256"]:
                raise RuntimeError(f"Original image changed after audit: {record['sample_id']}")
            cursor += 1
            records.append(record)
    if cursor != int(state["next_index"]):
        raise RuntimeError("Audit progress cursor does not equal verified committed records")
    _save_progress(progress_path, state)
    pending, first_index, last_save = [], cursor, time.monotonic()

    def commit():
        nonlocal pending, first_index, last_save
        if not pending:
            return
        filename = f"images-{first_index:06d}-{first_index + len(pending):06d}.json"
        shard = {"identity": identity, "first_index": first_index, "records": pending}
        atomic_json(folder / filename, shard)
        entry = {"file": filename, "sha256": file_sha256(folder / filename), "first_index": first_index, "count": len(pending)}
        state["shards"].append(entry)
        state["next_index"] = first_index + len(pending)
        state["status"] = "IN_PROGRESS"
        _save_progress(progress_path, state)
        records.extend(pending)
        first_index = state["next_index"]
        pending = []
        last_save = time.monotonic()

    for identifier in ids[cursor:]:
        original = dataset.path_for(identifier)
        record = {"sample_id": identifier, "label": int(dataset.index[identifier]["label"]),
                  "encoded_bytes": original.stat().st_size, "encoded_sha256": file_sha256(original)}
        try:
            item = dataset.get(identifier)
            image = item["image"]
            record.update(source_dimensions=item["source_dimensions"], source_mode=item["source_mode"],
                          rgb_sha256=hashlib.sha256(f"RGB:{image.width}:{image.height}:".encode() + image.tobytes()).hexdigest(),
                          status="DECODED")
        except Exception as exc:
            record.update(status="FAILED", error=repr(exc))
        pending.append(record)
        if len(pending) >= 250 or time.monotonic() - last_save >= 900:
            commit()
    commit()
    state["status"] = "COMPLETE"
    _save_progress(progress_path, state)
    groups = defaultdict(list)
    dimensions, modes, failures = Counter(), Counter(), []
    digest = hashlib.sha256()
    for record in records:
        split, _ = parse_sample_id(record["sample_id"])
        if record["status"] != "DECODED":
            failures.append({"sample_id": record["sample_id"], "error": record["error"]})
            continue
        groups[record["rgb_sha256"]].append(record["sample_id"])
        dimensions[f"{split}:{record['source_dimensions'][0]}x{record['source_dimensions'][1]}"] += 1
        modes[f"{split}:{record['source_mode']}"] += 1
        digest.update(f"{record['sample_id']}:{record['rgb_sha256']}\n".encode())
    duplicates = {key: value for key, value in groups.items() if len(value) > 1}
    conflicts = [{"rgb_sha256": key, "sample_ids": value,
                  "source_labels": [int(dataset.index[item]["label"]) for item in value]}
                 for key, value in duplicates.items() if len({dataset.index[item]["label"] for item in value}) > 1]
    audit = {"version": AUDIT_VERSION, "dataset_fingerprint": dataset.fingerprint,
        "images_attempted": len(ids), "images_decoded": len(ids) - len(failures), "failures": failures,
        "source_dimensions": dict(dimensions), "source_modes": dict(modes), "duplicate_groups": duplicates,
        "duplicate_group_count": len(duplicates), "duplicate_extra_rows": sum(len(value) - 1 for value in duplicates.values()),
        "duplicate_label_conflicts": conflicts, "duplicate_label_conflict_count": len(conflicts),
        "decoded_sequence_sha256": digest.hexdigest(), "original_record_shards": state["shards"],
        "scope": "all 101000 official original images, decoded RGB dimensions plus pixels before resizing",
        "limitations": "does not detect near duplicates, crops or semantic matches; labels remain as supplied",
        "declared_exclusion_rule": DUPLICATE_POLICY, "audit_checkpoint": str(progress_path)}
    audit["audit_sha256"] = canonical_hash(audit)
    path = dataset.cache / "decoded_rgb_audit.json"
    if path.exists() and json.loads(path.read_text()) != audit:
        raise RuntimeError("Reconstructed audit differs from committed final audit")
    atomic_json(path, audit)
    return audit


def construct_splits(dataset, audit, config):
    if audit["failures"] or audit["images_decoded"] != len(dataset):
        raise RuntimeError("Food-101 audit has decode failures; explicit protocol decision required")
    options = config.get("data", {})
    split_seed = int(options.get("split_seed", 314159))
    subset_seed = int(options.get("subset_seed", split_seed + 1))
    diagnostic_seed = int(options.get("diagnostic_seed", split_seed + 2))
    train_count = int(options.get("train_per_class", 100))
    dev_count = int(options.get("dev_per_class", 20))
    diagnostic_count = int(options.get("diagnostic_per_class", 2))
    reserve_count = int(options.get("reserve_dev_per_class", 100))
    if min(train_count, dev_count, diagnostic_count) < 1 or not diagnostic_count <= dev_count <= reserve_count < 750:
        raise ValueError("Require positive class counts, diagnostics <= selected dev <= reserved dev < 750")
    partitions = {"training": [], "development": [], "heldout": list(dataset.source_ids["test"])}
    for label in range(101):
        ids = sorted(record["sample_id"] for record in dataset.records["train"] if record["label"] == label)
        permutation = np.random.default_rng(stable_seed(split_seed, "official-train-split", label)).permutation(len(ids))
        chosen = [ids[int(i)] for i in permutation]
        partitions["development"].extend(chosen[:reserve_count])
        partitions["training"].extend(chosen[reserve_count:])
    membership = {identifier: part for part, ids in partitions.items() for identifier in ids}
    if len(membership) != len(dataset):
        raise RuntimeError("Candidate partitions do not cover each original identity exactly once")
    priority = {"heldout": 0, "development": 1, "training": 2}
    exclusions = []
    for digest, group in sorted(audit["duplicate_groups"].items()):
        ordered = sorted(group, key=lambda identifier: (priority[membership[identifier]], identifier))
        retained = ordered[0]
        for identifier in ordered[1:]:
            exclusions.append({"sample_id": identifier, "partition": membership[identifier],
                "retained_sample_id": retained, "retained_partition": membership[retained],
                "rgb_sha256": digest, "reason": "exact_original_decoded_RGB_duplicate"})
    excluded = {record["sample_id"] for record in exclusions}
    pools = {part: sorted(identifier for identifier in ids if identifier not in excluded) for part, ids in partitions.items()}

    def subset(ids, count, seed, stream):
        selected = []
        grouped = defaultdict(list)
        for identifier in ids:
            grouped[int(dataset.index[identifier]["label"])].append(identifier)
        for label in range(101):
            available = sorted(grouped[label])
            if len(available) < count:
                raise ValueError(f"Insufficient {stream} class {label} after duplicate policy: {len(available)} < {count}")
            permutation = np.random.default_rng(stable_seed(seed, stream, label)).permutation(len(available))
            selected.extend(available[int(i)] for i in permutation[:count])
        return selected

    selected = {"training": subset(pools["training"], train_count, subset_seed, "training-subset"),
                "development": subset(pools["development"], dev_count, subset_seed, "development-subset"),
                "heldout": pools["heldout"]}
    diagnostic_ids = subset(selected["development"], diagnostic_count, diagnostic_seed, "diagnostic-subset")
    if any(set(pools[a]) & set(pools[b]) for a, b in (("training", "development"), ("training", "heldout"), ("development", "heldout"))):
        raise AssertionError("Food-101 partition identities overlap")
    retained_ids = set().union(*map(set, pools.values()))
    if any(len(retained_ids.intersection(group)) != 1 for group in audit["duplicate_groups"].values()):
        raise AssertionError("Exact RGB duplicate exclusion policy was not fully applied")
    manifest = {"schema": DATA_SCHEMA, "dataset_fingerprint": dataset.fingerprint,
        "source_revision": dataset.manifest["source_revision"], "split_seed": split_seed, "subset_seed": subset_seed,
        "diagnostic_seed": diagnostic_seed, "reserve_dev_per_class": reserve_count,
        "candidate_pool_sizes": {part: len(ids) for part, ids in partitions.items()},
        "duplicate_policy": DUPLICATE_POLICY, "duplicate_audit_sha256": audit["audit_sha256"], "exclusions": exclusions,
        "pool_ids": pools, "pool_sizes": {part: len(ids) for part, ids in pools.items()},
        "pool_class_counts": {part: dict(Counter(str(dataset.index[item]["label"]) for item in ids)) for part, ids in pools.items()},
        "selected_ids": selected, "selected_sizes": {part: len(ids) for part, ids in selected.items()},
        "selected_per_class": {"training": train_count, "development": dev_count}, "diagnostic_ids": diagnostic_ids,
        "diagnostic_per_class": diagnostic_count, "sample_id_disjoint": True,
        "exact_RGB_cross_partition_duplicates_remaining": 0, "stratification_uses_labels": True,
        "probe_training_reuses_pretraining_pool": True, "pretraining_labels_policy": "labels excluded from JEPA loss arguments",
        "heldout_source_split": "test", "heldout_predictive_evaluation": "NOT_RUN; reserved"}
    manifest["split_sha256"] = canonical_hash(manifest)
    path = dataset.cache / ("split_" + manifest["split_sha256"][:16] + ".json")
    if path.exists() and json.loads(path.read_text()) != manifest:
        raise RuntimeError("Committed split manifest differs from deterministic reconstruction")
    atomic_json(path, manifest)
    _save_progress(Path(dataset.manifest["checkpoint_dir"]) / "split.json", {
        "schema": DATA_SCHEMA, "stage": "split", "status": "COMPLETE", "split_sha256": manifest["split_sha256"],
        "manifest_path": str(path), "dataset_fingerprint": dataset.fingerprint})
    return manifest


def prepare_food101(root, config, allow_download=False):
    options = config.get("data", {})
    if options.get("source_url", SOURCE_URL) != SOURCE_URL or options.get("archive_md5", SOURCE_MD5) != SOURCE_MD5:
        raise ValueError("Notebook02 uses the official Food-101 archive identity; register another protocol to change source")
    root = Path(root).resolve()
    with _one_writer(root / "data/food101"):
        cache, manifest, records = ensure_source(root, allow_download=allow_download)
        dataset = Food101Adapter(cache, manifest, records)
        audit = audit_source(dataset)
        split = construct_splits(dataset, audit, config)
    enriched = dict(manifest)
    enriched["integrity_audit"] = {key: value for key, value in audit.items() if key not in ("duplicate_groups", "original_record_shards")}
    enriched["integrity_audit"]["exclusion_summary"] = dict(Counter(record["partition"] + "->" + record["retained_partition"] for record in split["exclusions"]))
    enriched["integrity_audit_path"] = str(cache / "decoded_rgb_audit.json")
    return {"dataset": dataset, "dataset_manifest": enriched, "manifest": enriched,
            "split_manifest": split, "audit": audit, "fingerprint": dataset.fingerprint,
            "train_ids": split["selected_ids"]["training"], "dev_ids": split["selected_ids"]["development"],
            "heldout_ids": split["selected_ids"]["heldout"], "diagnostic_ids": split["diagnostic_ids"]}


## Deterministic images and generic spatial masks

Training augmentation and target/context sampling each derive independent stateless seeds from `(configured seed, sample ID, occurrence)`. Evaluation resizes the complete original RGB image without training augmentation. Spatial grids derive from `image_size / patch_size`; bounded retries enforce minimum context and target counts and record any fallback. No function below performs file I/O until explicitly called with a dataset.

In [ ]:
def preprocessing_descriptor(config=None):
    config = config or {}
    policy = config.get("preprocessing", {})
    resolution = int(config.get("image_size", 128))
    mean = list(map(float, policy.get("mean", [0.5, 0.5, 0.5])))
    std = list(map(float, policy.get("std", [0.5, 0.5, 0.5])))
    scale = list(map(float, policy.get("crop_scale", [0.6, 1.0])))
    ratio = list(map(float, policy.get("crop_ratio", [0.75, 4 / 3])))
    flip = float(policy.get("horizontal_flip", 0.5))
    if resolution < 1 or len(mean) != 3 or len(std) != 3 or min(std) <= 0 or not all(math.isfinite(x) for x in mean + std):
        raise ValueError("Require positive resolution and finite three-channel normalization with positive std")
    if len(scale) != 2 or not 0 < scale[0] <= scale[1] <= 1 or len(ratio) != 2 or not 0 < ratio[0] <= ratio[1] or not 0 <= flip <= 1:
        raise ValueError("Invalid crop or flip policy")
    return {"version": "food101-original-rgb/v1", "resolution": resolution,
            "training": {"crop_area": scale, "crop_aspect": ratio, "crop_attempts": 10,
                         "interpolation": "PIL bilinear", "horizontal_flip_probability": flip},
            "evaluation": "complete original RGB image; bilinear square resize; no random crop or flip",
            "normalization": {"mean": mean, "std": std, "learned_from_data": False}}


def preprocess_image(image, config=None, seed=0, training=False):
    descriptor = preprocessing_descriptor(config)
    resolution = descriptor["resolution"]
    image = image.convert("RGB")
    width, height = image.size
    original_dimensions = [width, height]
    generator = np.random.default_rng(int(seed))
    crop, flipped, fallback = [0, 0, width, height], False, False
    if training:
        for attempt in range(descriptor["training"]["crop_attempts"]):
            area = width * height * generator.uniform(*descriptor["training"]["crop_area"])
            aspect = math.exp(generator.uniform(*map(math.log, descriptor["training"]["crop_aspect"])))
            crop_width, crop_height = round(math.sqrt(area * aspect)), round(math.sqrt(area / aspect))
            if 0 < crop_width <= width and 0 < crop_height <= height:
                left = int(generator.integers(0, width - crop_width + 1))
                top = int(generator.integers(0, height - crop_height + 1))
                crop = [left, top, left + crop_width, top + crop_height]
                break
        else:
            side = min(width, height)
            left, top = (width - side) // 2, (height - side) // 2
            crop, fallback = [left, top, left + side, top + side], True
        image = image.crop(tuple(crop))
        flipped = bool(generator.random() < descriptor["training"]["horizontal_flip_probability"])
        if flipped:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    if image.size != (resolution, resolution):
        image = image.resize((resolution, resolution), Image.Resampling.BILINEAR)
    tensor = torch.from_numpy(np.asarray(image, dtype=np.uint8).copy()).permute(2, 0, 1).float().div_(255)
    mean = torch.tensor(descriptor["normalization"]["mean"])[:, None, None]
    std = torch.tensor(descriptor["normalization"]["std"])[:, None, None]
    tensor = (tensor - mean) / std
    metadata = {"seed": int(seed), "training": bool(training), "source_dimensions": original_dimensions,
                "crop_ltrb": crop, "flipped": flipped, "crop_fallback": fallback,
                "preprocessing_sha256": canonical_hash(descriptor)}
    return tensor, metadata


def _stream_seed(config, stream, default):
    seeds = config.get("seeds", {})
    aliases = {"augmentation": ("augmentation", "augment"), "mask": ("mask", "masking")}
    return int(next((seeds[key] for key in aliases.get(stream, (stream,)) if key in seeds), default))


def _mask_one(grid, seed, minimum_context=4, attempts=16, policy=None):
    policy = policy or {}
    rows, cols = map(int, grid)
    tokens = rows * cols
    minimum_context = int(policy.get("min_context", minimum_context))
    minimum_target = int(policy.get("min_target_tokens", 2))
    attempts = int(policy.get("whole_mask_attempts", attempts))
    target_count = int(policy.get("target_count", 4))
    if rows < 1 or cols < 1 or minimum_context < 1 or minimum_target < 1 or minimum_context + minimum_target > tokens or attempts < 1 or target_count < 1:
        raise ValueError("Invalid grid or bounded mask minimum/count contract")
    generator = torch.Generator().manual_seed(int(seed))
    chosen = None
    for retry in range(attempts):
        proposal = sample_masks((rows, cols), 1, generator,
            context_area=policy.get("context_area", (0.85, 1.0)), target_area=policy.get("target_area", (0.15, 0.20)),
            aspect_ratio=policy.get("aspect_ratio", (0.75, 1.5)), target_count=target_count,
            max_retries=int(policy.get("max_retries", 32)))
        if int(proposal["context_valid"].sum()) >= minimum_context and all(int(valid.sum()) >= minimum_target for valid in proposal["target_valid"]):
            chosen = proposal
            break
    fallback = chosen is None
    if fallback:
        # Select a deterministic contiguous top-left rectangle, and keep all
        # other positions as context. Targets may overlap each other by design.
        target_height = max(1, min(rows, int(math.sqrt(max(minimum_target, tokens * 0.15)))))
        target_width = max(1, min(cols, max(math.ceil(minimum_target / target_height), int(math.sqrt(max(minimum_target, tokens * 0.15))))))
        if target_height * target_width < minimum_target or tokens - target_height * target_width < minimum_context:
            candidates = [(h, w) for h in range(1, rows + 1) for w in range(1, cols + 1)
                          if minimum_target <= h * w <= tokens - minimum_context]
            if not candidates:
                raise ValueError("No rectangular target can satisfy configured mask minima")
            target_height, target_width = min(candidates, key=lambda pair: (abs(pair[0] * pair[1] / tokens - 0.15), pair))
        target = torch.tensor([[r * cols + c for r in range(target_height) for c in range(target_width)]], dtype=torch.long)
        hidden = set(target[0].tolist())
        context = torch.tensor([[index for index in range(tokens) if index not in hidden]], dtype=torch.long)
        chosen = {"context_indices": context, "context_valid": torch.ones_like(context, dtype=torch.bool),
                  "target_indices": [target.clone() for _ in range(target_count)],
                  "target_valid": [torch.ones_like(target, dtype=torch.bool) for _ in range(target_count)]}
    context = chosen["context_indices"][0][chosen["context_valid"][0]]
    blocks = [indices[0][valid[0]] for indices, valid in zip(chosen["target_indices"], chosen["target_valid"])]
    if len(context) < minimum_context or any(len(block) < minimum_target or bool(torch.isin(context, block).any()) for block in blocks):
        raise AssertionError("Context/target isolation or size contract violated")
    details = []
    for block in blocks:
        block_rows, block_cols = block // cols, block % cols
        details.append({"count": len(block), "height": int(block_rows.max() - block_rows.min() + 1),
                        "width": int(block_cols.max() - block_cols.min() + 1), "area_fraction": len(block) / tokens})
    chosen["metadata"] = {"seed": int(seed), "identity": canonical_hash({"grid": [rows, cols], "context": context.tolist(), "targets": [block.tolist() for block in blocks]}),
        "grid": [rows, cols], "context_count": len(context), "minimum_context": minimum_context,
        "minimum_target_tokens": minimum_target, "target_blocks": details, "whole_mask_retry": retry,
        "max_whole_mask_attempts": attempts, "fallback": fallback, "target_overlap_allowed": True}
    return chosen


def _pad_rows(rows):
    width = max(map(len, rows))
    indices = torch.full((len(rows), width), -1, dtype=torch.long)
    valid = torch.zeros_like(indices, dtype=torch.bool)
    for index, row in enumerate(rows):
        indices[index, :len(row)] = row
        valid[index, :len(row)] = True
    return indices, valid


def make_batch(dataset, sample_ids, occurrences, config, training=True):
    if not sample_ids or len(sample_ids) != len(occurrences):
        raise ValueError("Require nonempty paired image identities and occurrence numbers")
    resolution = int(config.get("image_size", 128))
    patch = int(config.get("model", {}).get("patch_size", config.get("patch_size", 8)))
    if patch < 1 or resolution < patch or resolution % patch:
        raise ValueError("Image size must be a positive integer multiple of patch size")
    grid = (resolution // patch, resolution // patch)
    target_count = int(config.get("masking", {}).get("target_count", 4))
    images, labels, augments, masks, mask_records = [], [], [], [], []
    for identifier, occurrence in zip(sample_ids, occurrences):
        if int(occurrence) < 0:
            raise ValueError("Occurrence must be nonnegative")
        item = dataset.get(identifier)
        augmentation_seed = stable_seed(_stream_seed(config, "augmentation", 1704), "augmentation", identifier, int(occurrence)) if training else 0
        image, augmentation = preprocess_image(item["image"], config, augmentation_seed, training)
        augmentation.update(sample_id=identifier, occurrence=int(occurrence))
        images.append(image)
        labels.append(int(item["label"]))
        augments.append(augmentation)
        mask_seed = stable_seed(_stream_seed(config, "mask", 1705), "mask", identifier, int(occurrence))
        mask = _mask_one(grid, mask_seed, policy=config.get("masking", {}))
        masks.append(mask)
        mask_records.append(mask["metadata"])
    context_indices, context_valid = _pad_rows([mask["context_indices"][0][mask["context_valid"][0]] for mask in masks])
    targets = [_pad_rows([mask["target_indices"][k][0][mask["target_valid"][k][0]] for mask in masks]) for k in range(target_count)]
    packed = {"context_indices": context_indices, "context_valid": context_valid,
              "target_indices": [item[0] for item in targets], "target_valid": [item[1] for item in targets],
              "metadata": {"examples": mask_records, "grid": list(grid), "target_overlap_allowed": True}}
    return {"images": torch.stack(images), "labels": torch.tensor(labels, dtype=torch.long), "sample_ids": list(sample_ids),
            "occurrences": list(map(int, occurrences)), "augmentation_ids": [canonical_hash(item) for item in augments],
            "mask_ids": [item["identity"] for item in mask_records], "augmentations": augments, "masks": packed,
            "mask_metadata": mask_records, "preprocessing": preprocessing_descriptor(config)}


def evaluation_batch(dataset, sample_ids, config=None):
    records = [dataset.get(identifier) for identifier in sample_ids]
    if not records:
        raise ValueError("Evaluation batch cannot be empty")
    return {"images": torch.stack([preprocess_image(record["image"], config, training=False)[0] for record in records]),
            "labels": torch.tensor([record["label"] for record in records], dtype=torch.long), "sample_ids": list(sample_ids)}


class EvaluationDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, sample_ids, config=None):
        self.dataset = dataset
        self.sample_ids = list(sample_ids)
        self.config = config or {}
        self.fingerprint = canonical_hash({"dataset": dataset.fingerprint, "sample_ids": self.sample_ids,
                                          "preprocessing": preprocessing_descriptor(self.config)})

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, index):
        record = self.dataset.get(self.sample_ids[index])
        return {"image": preprocess_image(record["image"], self.config, training=False)[0],
                "label": int(record["label"]), "sample_id": record["sample_id"]}


## Validation and use

This file has only static notebook/AST validation at authoring time. No archive was downloaded, no images decoded and no project function executed. Runtime claims remain **NOT_RUN** until the main Notebook 02 stages are deliberately launched.

In the main notebook, the first caller should use `prepare_food101(PROJECT_ROOT, config, allow_download=False)` with the default download setting. When intentionally enabled, the same entry point handles acquisition, stage resume and dataset auditing. Use `bundle['diagnostic_ids']` for the fixed development diagnostic panel. Training and probe functions must consume the Food-101 preprocessing definitions above, rather than Notebook 01's hardcoded 64×64 preprocessing.